<a href="https://colab.research.google.com/github/ZOUMKAI/MachineLearning/blob/main/0709_Colab_LINE_Bot_with_GEMINI_Rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 9.1 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://landing-skied-babbling.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://landing-skied-babbling.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。該校於1966年創立，前身為明新工業專科學校，歷經改制為明新技術學院，並於2002年正式升格為明新科技大學。

**歷史沿革**
明新科技大學的創校可追溯至1966年3月1日成立的明新工業專科學校，初期設有機械、土木、工業管理三科。隨著時代發展與教育需求，學校不斷擴展科系。1993年更名為明新工商專科學校，1997年改制為明新技術學院並附設專科部，最終於2002年9月奉教育部核准，正式升格為明新科技大學。2018年12月，更名為「明新學校財團法人明新科技大學」。

**辦學理念與校訓**
明新科大以《大學》中的「在明明德，在新民，在止於至善」為校名精義，旨在闡揚人類與生俱來的德性與情操，期許學子涵養高尚品德，具備專業學問與優良技術，以達全人發展的境界。其校訓為「堅毅、求新、創造」。學校願景為「深耕在地、放眼國際」，教育目標為「培養具實務經驗與人文素養之專業人才」。

**學院與科系**
目前明新科技大學設有半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院等六個學院。其中，半導體學院是學校積極發展的重點領域，設有半導體科技博士學位學程。其他學院則涵蓋多元領域，如：
*   **工程學院**：包含機械工程系、土木工程與環境資源管理系、資訊工程系、風力發電學士學位學程等。
*   **管理學院**：設有工業工程與管理系、資訊管理系、行銷與流通管理系、企業管理系、財務金融系等。
*   **民生學院**：包括旅館管理與廚藝創意系、幼兒保育系、休閒事業管理系、樂齡服務產業管理系等。
*   **人文與設計學院**：涵蓋國際商務外語系、運動管理系、多媒體與遊戲發展系、時尚造型與設計系等。

**學校特色**
明新科大致力於培養具備實務經驗與人文素養的專業人才，並與時俱進地掌握產業趨勢。學校發展出「MUST四大育才特色」，包括多元學習、全球視野、永續經營與技術創新，引導學生跨域學習，以成為炙手可熱的技職人才。學校積極推動產學合作，並在半導體、AI、元宇宙、風電綠能等前瞻產業領域投入大量資源，例如建置「半導體產業設備廠務與檢測人才培育基地」。此外，明新科大也重視國際化發展，擁有來自多個國家地區的國際

In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort
import logging
import os
import time
from google.genai import types

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    MessagingApiBlob,
    ReplyMessageRequest,
    TextMessage
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
    FileMessageContent
)

app = Flask(__name__)

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)
app.logger.setLevel(logging.INFO)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)

# 儲存檔案的目錄
UPLOAD_DIR = "/content/uploaded_files"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# 儲存每個使用者的對話 session 和上傳的檔案
user_sessions = {}  # {user_id: {"chat": chat_object, "uploaded_file": gemini_file}}

def get_user_session(user_id):
    """取得或建立使用者的對話 session"""
    if user_id not in user_sessions:
        # 建立新的對話 session
        new_chat = client.chats.create(
            model="gemini-2.5-flash",
            config=GenerateContentConfig(
                system_instruction="你是一個中文的AI助手，請用繁體中文回答。如果使用者有提供參考文件，請根據文件內容回答問題。",
                tools=[google_search_tool],
                response_modalities=["TEXT"],
            )
        )
        user_sessions[user_id] = {
            "chat": new_chat,
            "uploaded_file": None
        }
    return user_sessions[user_id]

def download_line_file(message_id, file_name):
    """從 LINE 下載使用者上傳的檔案"""
    with ApiClient(configuration) as api_client:
        line_bot_blob_api = MessagingApiBlob(api_client)
        file_content = line_bot_blob_api.get_message_content(message_id)

        file_path = os.path.join(UPLOAD_DIR, file_name)

        with open(file_path, 'wb') as f:
            f.write(file_content)

        return file_path

def upload_file_to_gemini(file_path):
    """上傳檔案到 Gemini Files API"""
    uploaded_file = client.files.upload(
        file=file_path,
        config={'display_name': os.path.basename(file_path)}
    )

    # 等待檔案處理完成
    while uploaded_file.state.name == "PROCESSING":
        print("檔案處理中...")
        time.sleep(1)
        uploaded_file = client.files.get(name=uploaded_file.name)

    if uploaded_file.state.name == "FAILED":
        raise Exception("檔案上傳處理失敗")

    return uploaded_file

def query_with_rag(user_id, question):
    """使用 RAG 模式回答問題"""
    session = get_user_session(user_id)
    uploaded_file = session["uploaded_file"]

    if uploaded_file:
        # 有上傳檔案，使用 RAG 模式
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=[
                types.Content(
                    role="user",
                    parts=[
                        types.Part.from_uri(
                            file_uri=uploaded_file.uri,
                            mime_type=uploaded_file.mime_type
                        ),
                        types.Part.from_text(text=f"請根據上述提供的檔案內容，用繁體中文回答這個問題：{question}")
                    ]
                )
            ]
        )
        return response.text
    else:
        # 沒有上傳檔案，使用一般多輪對話
        response = session["chat"].send_message(message=question)
        return response.text

@app.route("/", methods=['POST'])
def callback():
    signature = request.headers['X-Line-Signature']
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature.")
        abort(400)

    return 'OK'

@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    """處理文字訊息"""
    text = event.message.text
    user_id = event.source.user_id

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        if text.startswith('AI '):
            prompt = text[3:]
            try:
                # 使用 RAG 或一般對話
                reply_text = query_with_rag(user_id, prompt)

                # 檢查是否有上傳檔案，加上提示
                session = get_user_session(user_id)
                if session["uploaded_file"]:
                    reply_text = f"📄 [RAG 模式]\n\n{reply_text}"

                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=reply_text)]
                    )
                )
            except Exception as e:
                line_bot_api.reply_message_with_http_info(
                    ReplyMessageRequest(
                        reply_token=event.reply_token,
                        messages=[TextMessage(text=f"❌ 發生錯誤：{str(e)}")]
                    )
                )
        elif text == "清除文件":
            # 清除使用者上傳的檔案
            session = get_user_session(user_id)
            session["uploaded_file"] = None
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="✅ 已清除上傳的文件，恢復一般對話模式。")]
                )
            )
        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="請輸入「AI 問題」來開始對話\n或上傳 TXT/PDF 檔案啟用 RAG 模式")]
                )
            )

@handler.add(MessageEvent, message=FileMessageContent)
def handle_file_message(event):
    """處理使用者上傳的檔案"""
    user_id = event.source.user_id
    file_name = event.message.file_name

    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)

        # 檢查檔案類型
        if not (file_name.endswith('.txt') or file_name.endswith('.pdf')):
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text="⚠️ 目前只支援 TXT 或 PDF 檔案")]
                )
            )
            return

        try:
            # 下載檔案
            file_path = download_line_file(event.message.id, file_name)
            print(f"檔案已下載：{file_path}")

            # 上傳到 Gemini
            uploaded_file = upload_file_to_gemini(file_path)
            print(f"檔案已上傳到 Gemini：{uploaded_file.uri}")

            # 儲存到使用者 session
            session = get_user_session(user_id)
            session["uploaded_file"] = uploaded_file

            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"✅ 檔案「{file_name}」上傳成功！\n\n現在您可以輸入「AI 問題」來詢問關於這份文件的問題。\n\n輸入「清除文件」可恢復一般對話模式。")]
                )
            )
        except Exception as e:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=f"❌ 檔案處理失敗：{str(e)}")]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
INFO:__main__:Request body: {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615047547018281544","quoteToken":"8N9MuEuiJTgkdzqb9Q0Rq9ZAJHWrE28zcjpIcDzdN8cyu7r9F0WdmhxuR874ZU1Xb7HGxZg0rE1gNP813wUrfy3F0KMk2HwrbDx5JeKAElMn64PSa27LKFSNDoAMhUGtcKFb6lCPYzVNHhVZuZq0dg","markAsReadToken":"AlC0jZLYvUkgKjJCoJjZ-ZKfcFTMuGP7CJHz5ZzBYRxXyB2Ials-UAkhuX5-RcI38u1VkEKrDPGoGkkwFKWPBGC_7_4Ic1-5qfjske72u4kl6NftbY-HhZu2nMAKm2qGnm4F05RWBdZ0CIR2W4QuafrLYyIVaffUeDaoCaO5uILhjTKJRd2JzXJ9DP-V189-nvK6oLHYdnFlkKPshVBIoQ","text":"AI 校長愛吃什麼"},"webhookEventId":"01KS72ZDEJMPT2HKCC3QZNPEP6","deliveryContext":{"isRedelivery":false},"timestamp":1779428209813,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"ef02ec53c5b6400b8b25

BODY:  {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615047547018281544","quoteToken":"8N9MuEuiJTgkdzqb9Q0Rq9ZAJHWrE28zcjpIcDzdN8cyu7r9F0WdmhxuR874ZU1Xb7HGxZg0rE1gNP813wUrfy3F0KMk2HwrbDx5JeKAElMn64PSa27LKFSNDoAMhUGtcKFb6lCPYzVNHhVZuZq0dg","markAsReadToken":"AlC0jZLYvUkgKjJCoJjZ-ZKfcFTMuGP7CJHz5ZzBYRxXyB2Ials-UAkhuX5-RcI38u1VkEKrDPGoGkkwFKWPBGC_7_4Ic1-5qfjske72u4kl6NftbY-HhZu2nMAKm2qGnm4F05RWBdZ0CIR2W4QuafrLYyIVaffUeDaoCaO5uILhjTKJRd2JzXJ9DP-V189-nvK6oLHYdnFlkKPshVBIoQ","text":"AI 校長愛吃什麼"},"webhookEventId":"01KS72ZDEJMPT2HKCC3QZNPEP6","deliveryContext":{"isRedelivery":false},"timestamp":1779428209813,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"ef02ec53c5b6400b8b25117f1d1cbea6","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 05:36:56] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"file","id":"615047589465162188","markAsReadToken":"OicW5gIKyJSc69E1GokQVobAgX9XoaPdMoNpwynVFZmWr7SSnPlU_IddSChaEwh5yIYUdKc8L92V44ZHgYtI9MfvOqjmwi_imbCbhvqiTaOVqSZms-oi1cfW4ZNcUdSNaL3ZChBPIzJu4450EJXHB-FwkG6dfY3rU1W0XHmkij4iPAJKG7r2iJKFwhGMagmxEwuZPtVsgRJE9Ibji107pw","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KS7305VEDCNS13C0VZR635GS","deliveryContext":{"isRedelivery":false},"timestamp":1779428234972,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"3246d848140e4870bd7c2544826fd12a","mode":"active"}]}


BODY:  {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"file","id":"615047589465162188","markAsReadToken":"OicW5gIKyJSc69E1GokQVobAgX9XoaPdMoNpwynVFZmWr7SSnPlU_IddSChaEwh5yIYUdKc8L92V44ZHgYtI9MfvOqjmwi_imbCbhvqiTaOVqSZms-oi1cfW4ZNcUdSNaL3ZChBPIzJu4450EJXHB-FwkG6dfY3rU1W0XHmkij4iPAJKG7r2iJKFwhGMagmxEwuZPtVsgRJE9Ibji107pw","fileName":"RAG.txt","fileSize":21,"contentProvider":{"type":"line"}},"webhookEventId":"01KS7305VEDCNS13C0VZR635GS","deliveryContext":{"isRedelivery":false},"timestamp":1779428234972,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"3246d848140e4870bd7c2544826fd12a","mode":"active"}]}
檔案已下載：/content/uploaded_files/RAG.txt
檔案已上傳到 Gemini：https://generativelanguage.googleapis.com/v1beta/files/rqzyf7sw7tgb


INFO:werkzeug:127.0.0.1 - - [22/May/2026 05:37:17] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615047601980965256","quoteToken":"sFEk4fcOa7gvBY9JL7q50MkHypk4BL2H8VnBmq4coNpC_U26E3AzPUEC6BGgwyWhcyKnS4mWuHkjrjiQPJH7U1OkCNhAqtrz8vcLqDcXqgpEqhqCr-5ObN8huOgITTW1mhUXtfXJeEsT3T3D4wB38w","markAsReadToken":"Afxd4H9UA2rm5o3sGDhyuHdE_QlFU7S69C78zoIcyShif9u1Wa5YC7glTfSZ5tkBcXkQDUVy4WQLGKmKldQhTGYyu8xU7KhcCgCH9LmYOh5xfmG0LdL7CAMd41FCDa3W0L3BRx7ZtvGJunS05nqJJaie5OzGeqeWN2dvG3h6jOMM90Vhhu_ROCzZuCfCqqi6gH8Pgez7So6lZiFmvu1HdA","text":"校長愛吃什麼"},"webhookEventId":"01KS730D865E1VVH129TZSBGBF","deliveryContext":{"isRedelivery":false},"timestamp":1779428242371,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"f97deb338d0446cbb2cefe1507751ca8","mode":"active"}]}


BODY:  {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615047601980965256","quoteToken":"sFEk4fcOa7gvBY9JL7q50MkHypk4BL2H8VnBmq4coNpC_U26E3AzPUEC6BGgwyWhcyKnS4mWuHkjrjiQPJH7U1OkCNhAqtrz8vcLqDcXqgpEqhqCr-5ObN8huOgITTW1mhUXtfXJeEsT3T3D4wB38w","markAsReadToken":"Afxd4H9UA2rm5o3sGDhyuHdE_QlFU7S69C78zoIcyShif9u1Wa5YC7glTfSZ5tkBcXkQDUVy4WQLGKmKldQhTGYyu8xU7KhcCgCH9LmYOh5xfmG0LdL7CAMd41FCDa3W0L3BRx7ZtvGJunS05nqJJaie5OzGeqeWN2dvG3h6jOMM90Vhhu_ROCzZuCfCqqi6gH8Pgez7So6lZiFmvu1HdA","text":"校長愛吃什麼"},"webhookEventId":"01KS730D865E1VVH129TZSBGBF","deliveryContext":{"isRedelivery":false},"timestamp":1779428242371,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"f97deb338d0446cbb2cefe1507751ca8","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 05:37:23] "POST / HTTP/1.1" 200 -
INFO:__main__:Request body: {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615047622852083805","quoteToken":"7D8y8CeyKbEv9l2JE3hWdLNhJ1ryrfykvm-hviIqLSlMIck33usohTuySZX73eISqp_pAgHO0quR31VhqrMmLTWvSV67EOhKDWBrSYr2v-0SmezivC3SX--q96UXwiml5ZTy9mgc4_dk9B86fa8vww","markAsReadToken":"g63UMXUmp279wlY9HkZpqR66gnVnVeeg4M9T0_BXEDqM_BcwM2r6iJmbiU61_4fTAs4JIX9EGwyvb8f01O5NsR1iaxSnnS7-eybmHSBB6OrmV5nlJ8W_PMKgo3otn3bS0shNkOvS01Hp0YjYNcCKbgUr9QcBf_aBbxWVab8a4xTmssdbcEFB32fYKJh4duvl7CYbwNrLXUyROeMZh5xk9g","text":"AI 校長愛吃什麼"},"webhookEventId":"01KS730SAD53176SWGKGV5XP5P","deliveryContext":{"isRedelivery":false},"timestamp":1779428254953,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"0377ff0740ac445d85494c2a14d04733","mode":"active"}]}


BODY:  {"destination":"Uca511fbc529970b435eac4d5c87f9471","events":[{"type":"message","message":{"type":"text","id":"615047622852083805","quoteToken":"7D8y8CeyKbEv9l2JE3hWdLNhJ1ryrfykvm-hviIqLSlMIck33usohTuySZX73eISqp_pAgHO0quR31VhqrMmLTWvSV67EOhKDWBrSYr2v-0SmezivC3SX--q96UXwiml5ZTy9mgc4_dk9B86fa8vww","markAsReadToken":"g63UMXUmp279wlY9HkZpqR66gnVnVeeg4M9T0_BXEDqM_BcwM2r6iJmbiU61_4fTAs4JIX9EGwyvb8f01O5NsR1iaxSnnS7-eybmHSBB6OrmV5nlJ8W_PMKgo3otn3bS0shNkOvS01Hp0YjYNcCKbgUr9QcBf_aBbxWVab8a4xTmssdbcEFB32fYKJh4duvl7CYbwNrLXUyROeMZh5xk9g","text":"AI 校長愛吃什麼"},"webhookEventId":"01KS730SAD53176SWGKGV5XP5P","deliveryContext":{"isRedelivery":false},"timestamp":1779428254953,"source":{"type":"user","userId":"U4c28faa6b4288acea2070a1906546976"},"replyToken":"0377ff0740ac445d85494c2a14d04733","mode":"active"}]}


INFO:werkzeug:127.0.0.1 - - [22/May/2026 05:37:37] "POST / HTTP/1.1" 200 -
